# 可选实验 - 多类别分类


## 1.1 目标
在本实验中，你将探索一个用神经网络做多类别分类的例子。
<figure>
 <img src="./图片/C2_W2_mclass_header.png"   style="width500px;height:200px;">

> **图中文字（中文）**：多类别分类示例 P(y = 1|x)、P(y = 2|x)、P(y = 3|x)、P(y = 4|x) 左：二元分类（Binary）；右：多类别分类（Multiclass）
</figure>


## 1.2 工具
你将使用一些绘图程序。它们保存在本目录下的 `lab_utils_multiclass_TF.py` 中。

In [1]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib widget
from sklearn.datasets import make_blobs
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
np.set_printoptions(precision=2)
from lab_utils_multiclass_TF import *
import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)
tf.autograph.set_verbosity(0)

# 2.0 多类别分类
神经网络常用于对数据分类。例如，神经网络可以：
- 接收照片，把照片中的主体分类为 {狗, 猫, 马, 其他}
- 接收一个句子，把其中各成分的词性分类为：{名词, 动词, 形容词等}  

这类网络在最后一层会有多个单元。每个输出对应一个类别。当把一个输入样本送入网络时，取值最大的输出就是预测出的类别。如果把输出送入 softmax 函数，softmax 的输出将给出该输入属于各个类别的概率。

在本实验中，你将看到一个在 Tensorflow 中构建多类别网络的例子。随后我们将看看这个神经网络是如何做出预测的。

让我们先创建一个四类数据集。

## 2.1 准备并可视化数据
我们将使用 Scikit-Learn 的 `make_blobs` 函数生成一个含 4 个类别的训练数据集，如下图所示。

In [2]:
# 为分类创建四类别数据集
classes = 4
m = 100
centers = [[-5, 2], [-2, -2], [1, 2], [5, -2]]
std = 1.0
X_train, y_train = make_blobs(n_samples=m, centers=centers, cluster_std=std,random_state=30)

In [5]:
plt_mc(X_train,y_train,classes, centers, std=std)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

每个点代表一个训练样本。坐标轴 (x0,x1) 是输入，颜色表示该样本所属的类别。训练完成后，模型会接收一个新样本 (x0,x1)，并预测其类别。  

这个数据集虽然是生成的，却代表了许多真实的分类问题：有若干输入特征 (x0,...,xn) 和若干输出类别。模型被训练为使用输入特征预测正确的输出类别。

In [6]:
# 显示数据集中的类别
print(f"unique classes {np.unique(y_train)}")
# 展示类别的表示方式
print(f"class representation {y_train[:10]}")
# 显示数据集的形状
print(f"shape of X_train: {X_train.shape}, shape of y_train: {y_train.shape}")

unique classes [0 1 2 3]
class representation [3 3 3 0 3 3 3 3 2 0]
shape of X_train: (100, 2), shape of y_train: (100,)


## 2.2 模型
<img align="Right" src="./图片/C2_W2_mclass_lab_network.PNG"  style=" width:350px; padding: 10px 20px ; ">

> **图中文字（中文）**：网络结构：输入 x → 第 1 层（2 个单元）→ 第 2 层 → 输出概率 （softmax 单独应用，见下文）
本实验将使用如图所示的 2 层网络。
与二元分类网络不同，这个网络有四个输出，每个类别一个。给定一个输入样本，取值最大的输出就是该输入的预测类别。  

下面是一个在 Tensorflow 中构建该网络的例子。注意输出层使用的是 `linear` 而不是 `softmax` 激活函数。虽然也可以在输出层加入 softmax，但在训练时把线性输出传给损失函数数值上更稳定。如果要用模型预测概率，可以那时再应用 softmax。

In [7]:
tf.random.set_seed(1234)  # 为得到一致的结果而施加
model = Sequential(
    [
        Dense(2, activation = 'relu',   name = "L1"),
        Dense(4, activation = 'linear', name = "L2")
    ]
)

下面的语句编译并训练该网络。把 `from_logits=True` 作为损失函数的参数，表明输出激活函数是线性的而不是 softmax。

In [8]:
model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    optimizer=tf.keras.optimizers.Adam(0.01),
)

model.fit(
    X_train,y_train,
    epochs=200
)

Epoch 1/200
4/4 [==============================] - 0s 1ms/step - loss: 1.8158
Epoch 2/200
4/4 [==============================] - 0s 1ms/step - loss: 1.6976
Epoch 3/200
4/4 [==============================] - 0s 1ms/step - loss: 1.5989
Epoch 4/200
4/4 [==============================] - 0s 1ms/step - loss: 1.5179
Epoch 5/200
4/4 [==============================] - 0s 988us/step - loss: 1.4369
Epoch 6/200
4/4 [==============================] - 0s 1ms/step - loss: 1.3756
Epoch 7/200
4/4 [==============================] - 0s 1ms/step - loss: 1.3154
Epoch 8/200
4/4 [==============================] - 0s 1ms/step - loss: 1.2621
Epoch 9/200
4/4 [==============================] - 0s 1ms/step - loss: 1.2188
Epoch 10/200
4/4 [==============================] - 0s 1ms/step - loss: 1.1791
Epoch 11/200
4/4 [==============================] - 0s 1ms/step - loss: 1.1446
Epoch 12/200
4/4 [==============================] - 0s 997us/step - loss: 1.1129
Epoch 13/200
4/4 [==============================] - 0s 1m

4/4 [==============================] - 0s 1ms/step - loss: 0.1709
Epoch 105/200
4/4 [==============================] - 0s 1ms/step - loss: 0.1662
Epoch 106/200
4/4 [==============================] - 0s 1ms/step - loss: 0.1616
Epoch 107/200
4/4 [==============================] - 0s 1ms/step - loss: 0.1575
Epoch 108/200
4/4 [==============================] - 0s 1ms/step - loss: 0.1527
Epoch 109/200
4/4 [==============================] - 0s 1ms/step - loss: 0.1480
Epoch 110/200
4/4 [==============================] - 0s 1ms/step - loss: 0.1439
Epoch 111/200
4/4 [==============================] - 0s 1ms/step - loss: 0.1396
Epoch 112/200
4/4 [==============================] - 0s 1ms/step - loss: 0.1357
Epoch 113/200
4/4 [==============================] - 0s 1ms/step - loss: 0.1315
Epoch 114/200
4/4 [==============================] - 0s 1ms/step - loss: 0.1277
Epoch 115/200
4/4 [==============================] - 0s 1ms/step - loss: 0.1240
Epoch 116/200
4/4 [==============================] - 0

模型训练好后，我们可以看看它是如何对训练数据分类的。

In [9]:
plt_cat_mc(X_train, y_train, model, classes)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

上图中的决策边界展示了模型如何划分输入空间。这个非常简单的模型对训练数据分类毫无困难。它是怎么做到的？让我们更仔细地看看这个网络。

下面，我们将从模型中取出训练好的权重，用它绘制网络中每个单元的函数。再往下，有对结果的更详细解释。要成功使用神经网络并不需要了解这些细节，但更多地了解各层如何组合以解决分类问题，可能有助于建立直觉。

In [10]:
# 取出第一层训练好的参数
l1 = model.get_layer("L1")
W1,b1 = l1.get_weights()

In [11]:
# 绘制第一层的函数
plt_layer_relu(X_train, y_train.reshape(-1,), W1, b1, classes)

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [12]:
# 取出输出层训练好的参数
l2 = model.get_layer("L2")
W2, b2 = l2.get_weights()
# 创建“新特征”，即经过 L1 变换后的训练样本
Xl2 = np.zeros_like(X_train)
Xl2 = np.maximum(0, np.dot(X_train,W1) + b1)

plt_output_layer_linear(Xl2, y_train.reshape(-1,), W2, b2, classes,
                        x0_rng = (-0.25,np.amax(Xl2[:,0])), x1_rng = (-0.25,np.amax(Xl2[:,1])))

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

## 解释
#### 第 1 层 <img align="Right" src="./图片/C2_W2_mclass_layer1.png"  style=" width:600px; padding: 10px 20px ; ">

> **图中文字（中文）**：第 1 层单元 0：a = relu(W00·x0 + W01·x1) 第 1 层单元 1：a = relu(W10·x0 + W11·x1)
这些图展示了网络第一层中单元 0 和单元 1 的函数。坐标轴上的输入是 ($x_0,x_1$)。单元的输出用背景颜色表示，由每张图右侧的颜色条指示。注意，由于这些单元使用的是 ReLU，输出不一定落在 0 和 1 之间，在这个例子中其峰值大于 20。  
图中的等高线显示输出 $a^{[1]}_j$ 由零变为非零的转折点。回忆一下 ReLU 的图像：<img align="right" src="./图片/C2_W2_mclass_relu.png"  style=" width:200px; padding: 10px 20px ; ">

> **图中文字（中文）**：ReLU：g(z) = max(0, z) 当 z < 0 时 g(z) 为 0；当 z ≥ 0 时 g(z) = z 图中的等高线就是 ReLU 的拐点。

单元 0 把类别 0、1 与类别 2、3 分开。直线左侧的点（类别 0 和 1）输出为零，右侧的点输出大于零的值。  
单元 1 把类别 0、2 与类别 1、3 分开。直线上方的点（类别 0 和 2）输出为零，下方的点输出大于零的值。让我们看看这在下一层中会怎样！


#### 第 2 层，输出层  <img align="Right" src="./图片/C2_W2_mclass_layer2.png"  style=" width:600px; padding: 10px 20px ; ">

> **图中文字（中文）**：线性输出单元 0、1、2、3（四个散点图）

这些图中的点是经过第一层变换后的训练样本。一种理解方式是：第一层为第 2 层的评估创建了一组新的特征。这些图的坐标轴是上一层的输出 $a^{[1]}_0$ 和 $a^{[1]}_1$。正如上面所预测的，类别 0 和 1（绿色和蓝色）满足 $a^{[1]}_0 = 0$，而类别 1 和 2（蓝色和绿色）满足 $a^{[1]}_1 = 0$。  
同样，背景颜色的深浅表示取值的高低。  
单元 0 在 (0,0) 附近取得最大值，那里映射的是类别 0（蓝色）。    
单元 1 在左上角取得最大值，选中的是类别 1（绿色）。  
单元 2 指向右下角，那里是类别 3（橙色）。  
单元 3 在右上角取得最大值，选中我们的最后一个类别（紫色）。  

还有一个从图中不易看出的方面：各单元之间的取值是相互协调的。一个单元仅仅对它所要选中的类别产生最大值还不够，它还必须在该类别的点上比所有其他单元取值都高。这是由损失函数（`SparseCategoricalCrossEntropy`）中隐含的 softmax 函数完成的。与其他激活函数不同，softmax 作用于所有输出。

即使不了解每个单元在做什么的细节，你也可以成功使用神经网络。希望这个例子让你对“底层发生了什么”有了一些直觉。

## 恭喜！
你已经学会构建并运行用于多类别分类的神经网络。
